In [1]:
import duckdb
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path("Data")
DB_PATH = "olist.duckdb"

In [3]:
con = duckdb.connect(DB_PATH)
print("Connected Successfully!")

Connected Successfully!


In [4]:
customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
translation = pd.read_csv(DATA_PATH / "product_category_name_translation.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
geo = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")

In [5]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Items": items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Translation": translation,
    "Sellers": sellers,
    "Geolocation": geo
}
for name, df in datasets.items():
    print("="*60)
    print(name)
    print(df.shape)
    print(df.head())

Customers
(99441, 5)
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  
Orders
(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef4

In [6]:
con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM read_csv_auto('Data/olist_customers_dataset.csv');
""")

In [7]:
tables = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "translation": translation,
    "sellers": sellers,
    "geo": geo
}
for table_name, df in tables.items():
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df")

In [8]:
con.execute("SHOW TABLES").fetchdf()

,name
0,customer_features
1,customers
2,geo
3,items
4,master_orders
5,order_features
6,orders
7,payments
8,product_lookup
9,products


In [9]:
for table in tables.keys():
    print(f"\n{table}")
    result = con.execute(f"""
    SELECT *
    FROM {table}
    LIMIT 5
    """).fetchdf()
    print(result)


customers
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

orders
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186

In [10]:
for table in tables.keys():

    rows = con.execute(f"""

    SELECT COUNT(*)

    FROM {table}

    """).fetchone()[0]

    print(table, rows)

customers 99441
orders 99441
items 112650
payments 103886
reviews 99224
products 32951
translation 71
sellers 3095
geo 1000163


In [11]:
con.execute("""
CREATE OR REPLACE TABLE product_lookup AS
SELECT
p.product_id,t.product_category_name_english,
p.product_name_lenght,
p.product_description_lenght,
p.product_photos_qty,
p.product_weight_g,
p.product_length_cm,
p.product_height_cm,
p.product_width_cm
FROM products p
LEFT JOIN translation t
ON p.product_category_name =
t.product_category_name;
""")

In [12]:
con.execute("""
CREATE OR REPLACE TABLE master_orders AS
SELECT
o.order_id,
c.customer_id,
c.customer_unique_id,
c.customer_city,
c.customer_state,
o.order_status,
o.order_purchase_timestamp,
o.order_approved_at,
o.order_delivered_carrier_date,
o.order_delivered_customer_date,
o.order_estimated_delivery_date,
i.order_item_id,
i.product_id,
pl.product_category_name_english,
i.seller_id,
i.shipping_limit_date,
i.price,
i.freight_value,
pay.payment_sequential,
pay.payment_type,
pay.payment_installments,
pay.payment_value,
r.review_score,
r.review_comment_title,
r.review_comment_message,
r.review_creation_date,
r.review_answer_timestamp
FROM orders o
JOIN customers c
ON o.customer_id=c.customer_id
LEFT JOIN items i
ON o.order_id=i.order_id
LEFT JOIN product_lookup pl
ON i.product_id=pl.product_id
LEFT JOIN payments pay
ON o.order_id=pay.order_id
LEFT JOIN reviews r
ON o.order_id=r.order_id;
""")

In [13]:
master = con.execute("""
SELECT *
FROM master_orders
LIMIT 10
""").fetchdf()
master

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,...,8.72,2,voucher,1,18.59,4,None,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,...,22.76,1,boleto,1,141.46,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,...,19.22,1,credit_card,3,179.12,5,None,None,2018-08-18 00:00:00,2018-08-22 19:07:58
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,...,8.72,1,credit_card,1,28.62,5,None,None,2018-02-17 00:00:00,2018-02-18 13:02:51
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,...,27.36,1,credit_card,6,175.26,4,None,None,2017-07-27 00:00:00,2017-07-27 22:48:30
5,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,36edbb3fb164b1f16485364b6fb04c73,santa rosa,RS,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,None,None,...,16.05,1,credit_card,1,65.95,2,None,fiquei triste por n ter me atendido.,2017-05-13 00:00:00,2017-05-13 20:25:42
6,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,...,15.17,1,credit_card,3,75.16,5,None,None,2017-05-27 00:00:00,2017-05-28 02:59:57
7,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,...,16.05,1,boleto,1,35.95,1,None,None,2017-02-03 00:00:00,2017-02-05 01:58:35
8,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,...,19.77,1,credit_card,1,8.34,5,None,None,2017-08-17 00:00:00,2017-08-18 01:47:32
9,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,...,30.53,1,credit_card,1,259.06,1,None,Aguardando retorno da loja,2017-05-30 00:00:00,2017-05-30 23:13:47


In [14]:
con.execute("""
CREATE OR REPLACE TABLE master_orders AS
SELECT
    order_id,
    customer_id,
    customer_unique_id,
    customer_city,
    customer_state,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
    CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
    order_item_id,
    product_id,
    product_category_name_english,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    price,
    freight_value,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value,
    review_score,
    review_comment_title,
    review_comment_message,
    CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp
FROM master_orders;
""")

In [15]:
con.execute("""
DESCRIBE master_orders
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,customer_unique_id,VARCHAR,YES,None,None,None
3,customer_city,VARCHAR,YES,None,None,None
4,customer_state,VARCHAR,YES,None,None,None
5,order_status,VARCHAR,YES,None,None,None
6,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
7,order_approved_at,TIMESTAMP,YES,None,None,None
8,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
9,order_delivered_customer_date,TIMESTAMP,YES,None,None,None


In [16]:
master_df = con.execute("""
SELECT *
FROM master_orders
""").fetchdf()
master_df.to_csv("master_orders.csv", index=False)

In [17]:
con.close()
print("Database saved successfully.")

Database saved successfully.
